# TIP Environment Specifics and Claude Code Installation

Interactive exploration of the EPO Technology Intelligence Platform (TIP) runtime environment.

**Part of EPO Academy Training Material provided by mtc.berlin/depa.tech in March 2026**

Run the cells below to inspect your own container.

## 1. User identity & home directory

TIP uses `jovyan` (Jupyter convention) as the base user. A symlink maps your actual username to it.

In [8]:
import os, subprocess

print(f"whoami:    {os.environ.get('USER', subprocess.getoutput('whoami'))}")
print(f"$HOME:     {os.environ['HOME']}")
print(f"realpath:  {os.path.realpath(os.environ['HOME'])}")
print()
# Show the symlink
!ls -la /home/ | grep -v '^\.\.'

whoami:    arne.krueger
$HOME:     /home/arne.krueger
realpath:  /home/jovyan

total 12
drwxr-xr-x  1 root         root  4096 Mar 18 09:19 .
drwxr-xr-x  1 root         root  4096 Mar 18 09:19 ..
lrwxrwxrwx  1 root         root    12 Mar 18 09:19 arne.krueger -> /home/jovyan
drwx------ 40 arne.krueger users 4096 Mar 18 11:52 jovyan


## 2. Filesystem mounts

Only `/home/jovyan` is persistent (shown as `/` in JupyterLab's file browser).
Everything else — `/opt/conda`, the actual system root — is an **overlay** rebuilt from the container image on every restart.

In [9]:
import pandas as pd
import subprocess

# Parse mount info for key paths
rows = []
for target in ["/home/jovyan", "/home/jovyan/training", "/home/jovyan/.cache", "/opt/conda", "/"]:
    result = subprocess.run(
        ["findmnt", "--target", target, "-n", "-o", "TARGET,SOURCE,FSTYPE"],
        capture_output=True, text=True
    )
    if result.stdout.strip():
        parts = result.stdout.strip().split(None, 2)
        source = parts[1] if len(parts) > 1 else "?"
        fstype = parts[2] if len(parts) > 2 else "?"
        # Shorten kubernetes paths for readability
        if "kubernetes.io" in source:
            source = "EmptyDir (Kubernetes)"
        elif source.startswith("/dev/"):
            source = source.split("[")[0]
        persistent = "Yes" if parts[0] == "/home/jovyan" and fstype != "overlay" else "No"
        if parts[0] == "/home/jovyan":
            persistent = "Yes"
        rows.append({"Mount": parts[0], "Source": source, "Type": fstype, "Persistent": persistent})

df = pd.DataFrame(rows)
df.style.set_caption("Container mount points")

,Mount,Source,Type,Persistent
0,/home/jovyan,/dev/sdf,ext4,Yes
1,/home/jovyan/training,EmptyDir (Kubernetes),ext4,No
2,/home/jovyan/.cache,EmptyDir (Kubernetes),ext4,No
3,/,overlay,overlay,No
4,/,overlay,overlay,No


## 3. Training materials (Git worktree mount)

The `~/training/` directory is mounted from a **Kubernetes EmptyDir** populated via a Git worktree checkout at pod startup. All files are owned by UID 65533, not your user. The directory is read-only.

In [10]:
# Show the mount source (reveals the Git worktree path)
!findmnt --target ~/training -o SOURCE -n

print()

# Prove it's read-only in practice
import tempfile, os
test_path = os.path.expanduser("~/training/.write_test")
try:
    open(test_path, "w").close()
    os.remove(test_path)
    print("training/ is WRITABLE (unexpected!)")
except PermissionError:
    print("training/ is READ-ONLY (as expected)")

print()

# List top-level contents with ownership
!ls -la ~/training/

/dev/sda1[/var/lib/kubelet/pods/c3cfc213-e310-4681-9658-8888b9b7630f/volumes/kubernetes.io~empty-dir/training/.worktrees/6593d0cc2512f2d2ca82adab30d2eb0447dde420/training]

training/ is READ-ONLY (as expected)

total 20
drwxr-xr-x  5        65533 users 4096 Mar 18 09:19  .
drwx------ 40 arne.krueger users 4096 Mar 18 11:52  ..
drwxr-sr-x  2        65533 users 4096 Mar 18 09:19 'EP full-text data'
drwxr-sr-x  4        65533 users 4096 Mar 18 09:19 'Patent information others'
drwxr-sr-x  6        65533 users 4096 Mar 18 09:19  PATSTAT


## 4. Startup scripts & dotfile persistence

The init script `/usr/bin/init_juser.sh` runs on **every container start** and overwrites
`.bashrc`, `.profile`, and `.condarc` with EPO defaults. Any customizations in those files are lost.

**Rule of thumb:** Put all shell customizations in `.bash_aliases` — it is never touched by init scripts.

In [11]:
import os, datetime

# Check which dotfiles are overwritten on startup vs. persistent
dotfiles = [
    ".bashrc", ".profile", ".condarc",          # overwritten by init
    ".bash_aliases", ".gitconfig", ".npmrc",     # persistent
    ".ssh", ".claude", ".claude.json",           # persistent
]

# Container start time = .bashrc modification time (since it's always overwritten)
boot_time = os.path.getmtime(os.path.expanduser("~/.bashrc"))
boot_str = datetime.datetime.fromtimestamp(boot_time).strftime("%Y-%m-%d %H:%M:%S")
print(f"Container start time (from .bashrc): {boot_str}\n")

print(f"{'File':<20} {'Modified':<22} {'Overwritten on start?'}")
print("-" * 65)
for f in dotfiles:
    path = os.path.expanduser(f"~/{f}")
    if os.path.exists(path):
        mtime = os.path.getmtime(path)
        mtime_str = datetime.datetime.fromtimestamp(mtime).strftime("%Y-%m-%d %H:%M:%S")
        overwritten = "YES — lost on restart!" if abs(mtime - boot_time) < 60 else "No — persistent"
        print(f"{f:<20} {mtime_str:<22} {overwritten}")
    else:
        print(f"{f:<20} {'(not found)':<22}")

Container start time (from .bashrc): 2026-03-18 09:19:25

File                 Modified               Overwritten on start?
-----------------------------------------------------------------
.bashrc              2026-03-18 09:19:25    YES — lost on restart!
.profile             2026-03-18 09:19:25    YES — lost on restart!
.condarc             2026-03-18 09:19:25    YES — lost on restart!
.bash_aliases        2026-03-18 11:32:25    No — persistent
.gitconfig           2025-06-05 11:22:28    No — persistent
.npmrc               2026-03-18 12:03:55    No — persistent
.ssh                 2024-11-23 20:09:37    No — persistent
.claude              2026-03-18 12:08:57    No — persistent
.claude.json         2026-03-18 11:07:17    No — persistent


## 5. Python environment & TIP library access

The EPO TIP library (`epo.tipdata.patstat`) is installed in the base conda environment at `/opt/conda/...`.
When working inside a project venv, this library is invisible unless you explicitly bridge the gap.

In [12]:
import sys

print(f"Python:       {sys.executable}")
print(f"sys.prefix:   {sys.prefix}")
print(f"base_prefix:  {sys.base_prefix}")
print(f"In a venv:    {sys.prefix != sys.base_prefix}")
print()

# Check if TIP library is available
try:
    import epo.tipdata.patstat as tip
    print(f"epo.tipdata.patstat: {tip.__file__}")
except ImportError as e:
    print(f"epo.tipdata.patstat: NOT AVAILABLE ({e})")
    print()
    print("Fix options:")
    print("  1. python -m venv --system-site-packages .venv")
    print('  2. echo "/opt/conda/lib/python3.12/site-packages" > .venv/lib/python3.12/site-packages/conda.pth')

Python:       /opt/conda/bin/python
sys.prefix:   /opt/conda
base_prefix:  /opt/conda
In a venv:    False

epo.tipdata.patstat: /opt/conda/lib/python3.12/site-packages/epo/tipdata/patstat/__init__.py


## 6. Installing Claude Code persistently

`npm install -g` writes to `/opt/conda/` (overlay) by default — lost on restart.
The fix: redirect npm's global prefix to a persistent directory in your home.

Run the cell below for the **initial install** (one-time only, via npm — no Homebrew available on TIP).
After that, use `claude upgrade` to update.

In [13]:
%%bash
set -e

# 1. Create persistent directory
mkdir -p ~/.npm-global

# 2. Set npm prefix (skip if already configured)
if ! grep -q 'npm-global' ~/.npmrc 2>/dev/null; then
    npm config set prefix ~/.npm-global
    echo "Set npm prefix to ~/.npm-global"
else
    echo "npm prefix already configured — skipping"
fi

# 3. Add to PATH in .bash_aliases (idempotent — skips if already present)
if ! grep -q 'npm-global' ~/.bash_aliases 2>/dev/null; then
    sed -i '1i # Persistent npm global packages (survives container restarts)\nexport PATH="$HOME/.npm-global/bin:$PATH"\n' ~/.bash_aliases
    echo "Added PATH entry to ~/.bash_aliases"
else
    echo "PATH entry already in ~/.bash_aliases — skipping"
fi

# 4. Install Claude Code
export PATH="$HOME/.npm-global/bin:$PATH"
npm install -g @anthropic-ai/claude-code

# 5. Verify
echo ""
echo "Installed at: $(which claude)"
claude --version

npm prefix already configured — skipping
PATH entry already in ~/.bash_aliases — skipping

added 14 packages, and changed 3 packages in 1s

Installed at: /home/arne.krueger/.npm-global/bin/claude
2.1.78 (Claude Code)


## 7. Disk usage overview

Check how much space you're using on your persistent volume vs the ephemeral overlay.

In [14]:
!echo "=== Your persistent volume ===" && df -h /home/jovyan
!echo ""
!echo "=== Overlay (ephemeral) ===" && df -h /
!echo ""
!echo "=== Top directories by size in your home ===" && du -sh ~/*/  2>/dev/null | sort -rh | head -15

=== Your persistent volume ===
Filesystem      Size  Used Avail Use% Mounted on
/dev/sdf         30G   20G  9.9G  67% /home/jovyan

=== Overlay (ephemeral) ===
Filesystem      Size  Used Avail Use% Mounted on
overlay          95G   21G   75G  22% /

=== Top directories by size in your home ===
5.4G	/home/arne.krueger/medtech-analysis/
1.7G	/home/arne.krueger/mtc-patent-analytics/
606M	/home/arne.krueger/patlib/
546M	/home/arne.krueger/safe_backup/
471M	/home/arne.krueger/epo_pkf2024/
350M	/home/arne.krueger/fwpc/
290M	/home/arne.krueger/mtc-patstat-mcp/
196M	/home/arne.krueger/piznet/
182M	/home/arne.krueger/depa.tech/
181M	/home/arne.krueger/tip4patlibs/
79M	/home/arne.krueger/ping-vortrag/
58M	/home/arne.krueger/training/
53M	/home/arne.krueger/epo-codefest/
44M	/home/arne.krueger/patintelli/
38M	/home/arne.krueger/familytree/
